# 数据库正传：SQLite 与 SQL

## 数据库到底是什么？

数据库（database）不是某一种文件格式，而是一套管理数据的软件系统。它通常负责：

| 能力 | 作用 |
| --- | --- |
| 持久化 | 让进程停止后数据仍然保留 |
| 查询 | 按条件找出需要的记录 |
| 增删改 | 插入、删除和更新数据 |
| 事务 | 让一组操作要么全部成功，要么全部回滚 |
| 并发控制 | 多个请求同时读写时保持一致 |
| 索引 | 让常见查询不必扫描全部数据 |

## 先认识两组分类

数据库常从两个维度来分类。理解分类，是根据项目需求做选择的基础。

### 关系型与非关系型

**关系型数据库**用表（table）、行（row）和列（column）组织数据。字段结构明确，表之间可以建立关系，并使用 SQL 查询。SQLite、MySQL、PostgreSQL 都属于关系型数据库。

**非关系型数据库**不要求所有数据都服从固定的表结构，常见形式有文档、键值、列族和图数据库。例如 MongoDB 常用 JSON 风格文档，Redis 常用键值结构。

### 嵌入式与服务式

另一组分类看数据库如何运行：

| 类型 | 工作方式 | 例子 |
| --- | --- | --- |
| 嵌入式数据库 | 数据库引擎直接运行在应用进程中，数据通常是本地文件 | SQLite |
| 服务式数据库 | 独立启动一个数据库服务，应用通过网络连接 | MySQL、PostgreSQL、MongoDB |

## 结构化、半结构化、非结构化

还可以按照数据本身的形状来理解：

- **结构化数据**：（关系型数据库）字段和类型预先确定，例如电影的片名、语言和上映日期。
- **半结构化数据**：（非关系型数据库）有一定结构，但字段不完全固定，例如 JSON、XML。
- **非结构化数据**：（对象存储）主要是原始内容，例如图片、视频、音频和长篇文档。

## 在 REPL 中打开 SQLite

先进入项目环境中的 Python：

~~~bash
uv run python
~~~

导入标准库并连接数据库：

~~~pycon
>>> import sqlite3
>>> conn = sqlite3.connect("test.db")
>>> cur = conn.cursor()
~~~

`connect` 会打开 `test.db`。文件不存在时，SQLite 会自动创建它；`cur` 是执行 SQL 和读取结果的对象。连接结束后记得关闭：

~~~pycon
>>> conn.close()
~~~

## 建表：CREATE TABLE

一张表可以理解为一类记录的集合。

~~~sql
CREATE TABLE 表名 (字段名 字段类型, 字段名 字段类型, ……)
~~~

为电影历史记录建表：

~~~sql
CREATE TABLE films (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT,
    language TEXT,
    release_date TEXT,
    created_at TEXT
);
~~~

这里的每一列都有一个名字和类型：

- `id` 是每条记录的唯一编号。`PRIMARY KEY` 表示主键，`AUTOINCREMENT` 让 SQLite 自动生成递增编号。
- `title` 保存片名，`language` 保存语言。
- `release_date` 和 `created_at` 暂时用文字保存日期。

如果表可能已经存在，可以写成 `CREATE TABLE IF NOT EXISTS films (...)`，避免重复建表时报错。

### python 里执行 SQL 语句

需要用 python 的 cur 帮我们执行，所以在REPL中执行的时候可以写成下面这样，把SQL语句包在cur.execute()之中：

~~~sql
cur.execute("""
CREATE TABLE films (
    id           INTEGER PRIMARY KEY AUTOINCREMENT,
    title        TEXT,
    language     TEXT,
    release_date TEXT,
    created_at   TEXT
)
""")
~~~

敲回车后 REPL 会回显一个 Cursor 对象，不用管它。如果执行没有报错，这张 films 表就写进了刚才家目录里那个 test.db。

## SQLite 的五种常见类型

SQLite 的类型系统比很多数据库更宽松，常用存储类型有：

| 类型 | 用途 |
| --- | --- |
| `INTEGER` | 整数、计数、布尔值（通常用 0/1） |
| `REAL` | 浮点数，例如情感分数 |
| `TEXT` | 文本、日期时间字符串 |
| `BLOB` | 原始二进制数据 |
| `NULL` | 没有值 |

SQLite 没有专门的日期类型，项目中常把日期保存为 ISO 8601 格式的 `TEXT`。布尔值也通常使用 `INTEGER` 的 `0` 和 `1` 表示。

## 插入数据：INSERT 与 commit

~~~sql
INSERT INTO 表名 (字段名, 字段名, 字段名, ……) VALUES (值, 值, 值, ……)
~~~

建表后插入一条电影：

~~~pycon
>>> cur.execute(
...     "INSERT INTO films (title, language, release_date, created_at)"
...     "VALUES ('千与千寻', '日语', '2001-07-20', datetime('now'))"
... )
>>> conn.commit()
~~~

`INSERT INTO` 指定要写入的表和列，`VALUES` 提供对应的值。`datetime('now')` 由 SQLite 生成当前 UTC 时间。

`execute` 只是把修改放进当前事务，真正提交修改要调用 `conn.commit()`。如果忘记提交，程序结束或连接关闭时，插入可能不会保留下来。

## 查询数据：SELECT

~~~sql
SELECT 字段名1，字段名2 FROM 表名 WHERE 条件表达式
~~~

查询刚才写入的记录：

~~~pycon
>>> cur.execute("SELECT * FROM films").fetchall()
[(1, '千与千寻', '日语', '2001-07-20', '2026-09-07 08:00:00')]
~~~

`SELECT *` 表示选择全部列，不写 WHERE 不加任何条件，`fetchall()` 取回全部结果。也可以只选择需要的列：

SQL 关键字通常不区分大小写，但统一使用大写能让查询更容易阅读。

## 修改和删除：UPDATE、DELETE、DROP

删除一条记录：

~~~sql
DELETE FROM 表名 WHERE 条件表达式
~~~

修改一天记录

~~~sql
UPDATE 表名 SET 字段名 = 值 WHERE 条件表达式
~~~

修改执行后都要 `conn.commit()`来提交事务。

如果要删除整张表及其结构，使用：

~~~sql
DROP TABLE 表名
~~~

`DROP TABLE` 是破坏性操作，练习时可以使用，真实项目中必须确认目标和备份。

## 不要直接把用户输入拼进 SQL

假设搜索接口把用户输入直接拼接到 SQL 中：

当用户输入普通片名时看起来没问题，但输入 `' OR '1'='1` 后，查询条件可能变成永远为真的表达式：

~~~sql
SELECT * FROM films WHERE title = '' OR '1'='1';
~~~

这就是 SQL 注入（SQL injection）。攻击者可能借此读出不该读的数据，甚至修改或删除数据。字符串格式化和 f-string 都不能用来处理 SQL 参数。

## 使用 `?` 占位符防止 SQL 注入

~~~python
cursor.execute(
    'SELECT * FROM films WHERE title = ?',(title,),
)
rows = cursor.fetchall()
~~~

把值的部分写成 ? ，然后让 python 帮我们往这个SQL语句的问号上传值

只有一个参数时，`(title,)` 末尾的逗号不能省略：它表示只有一个元素的元组。多个参数依次写入：

~~~python
cursor.execute(
    'SELECT * FROM films WHERE language = ? AND release_date >= ?',
    ('英语', '2020-01-01'),
)
~~~

## 排序与限制：ORDER BY、DESC、LIMIT

### 新建 `seed_data.py` 脚本批量准备数据

~~~python
import sqlite3
import time

conn = sqlite3.connect("test.db")
cur = conn.cursor()
cur.execute("DROP TABLE IF EXISTS films")             # 清掉刚才手玩的，从头来
cur.execute("""
CREATE TABLE films (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT, language TEXT, release_date TEXT, created_at TEXT
)
""")

films = [
    ("肖申克的救赎", "英语", "1994-09-23"),
    ("阿甘正传", "英语", "1994-07-06"),
    ("霸王别姬", "汉语", "1993-01-01"),
    ("龙猫", "日语", "1988-04-16"),
    ("天空之城", "日语", "1986-08-02"),
    ("天堂电影院", "意大利语", "1988-11-17"),
    ("泰坦尼克号", "英语", "1997-12-19"),
    ("楚门的世界", "英语", "1998-06-05"),
    ("花样年华", "汉语", "2000-09-29"),
    ("千与千寻", "日语", "2001-07-20"),
    ("无间道", "汉语", "2002-12-12"),
    ("盗梦空间", "英语", "2010-07-16"),
    ("让子弹飞", "汉语", "2010-12-16"),
    ("少年派的奇幻漂流", "英语", "2012-11-21"),
    ("星际穿越", "英语", "2014-11-07"),
    ("疯狂动物城", "英语", "2016-03-04"),
    ("你的名字", "日语", "2016-08-26"),
    ("摔跤吧！爸爸", "印地语", "2016-12-23"),
    ("燃烧", "韩语", "2018-05-17"),
    ("寄生虫", "韩语", "2019-05-30"),
    ("' OR '1'='1", "英语", "2020-01-01"),      # 疯狂导演的“怪片名”
    ("奥德赛", "英语", "2026-07-17"),
    ("牛来", "汉语", "2026-08-05"),
    ("欢迎来龙餐馆", "汉语", "2026-08-11"),
]
for i, (title, language, release_date) in enumerate(films, 1):
    cur.execute(
        "INSERT INTO films (title, language, release_date, created_at) "
        "VALUES (?, ?, ?, datetime('now'))",           # created_at 取此刻的时间
        [title, language, release_date],
    )
    print(f"已灌入 {i}/{len(films)}：{title}")
    time.sleep(1)                                       # 歇 1 秒再插下一部，好让每部的入库时间错开
conn.commit()
print("完成")

~~~

保存脚本为 `seed_data.py` 后运行：

~~~bash
python seed_data.py
~~~

重新进入 Python REPL，连接同一个 `test.db`，依次测试：

~~~pycon
>>> import sqlite3
>>> conn = sqlite3.connect("test.db")
>>> cur = conn.cursor()
>>> cur.execute("SELECT id, title, created_at FROM films ORDER BY created_at").fetchall()
>>> cur.execute("SELECT id, title, created_at FROM films ORDER BY created_at DESC").fetchall()
>>> cur.execute("SELECT id, title, created_at FROM films ORDER BY created_at DESC LIMIT 5").fetchall()
~~~

三条查询分别表示按时间正序、按时间倒序，以及倒序后只取最新 5 条。

## 文件版和数据库版的对比

文件版接口大致是：

~~~python
records = load_history()   # 全读进内存
records.reverse()          # 自己倒叙
return records[:10]        # 自己切片
~~~

数据库版只需描述想要什么：

~~~python
cur.execute(
    'SELECT * FROM films ORDER BY created_at DESC LIMIT 10'
)
return cursor.fetchall()
~~~

数据库可以在底层决定怎样读取、怎样排序，并尽量只处理需要的部分。

## 数据库怎样避免内存爆炸？

面对数百万条记录，`ORDER BY` 似乎也要先读完全部数据。以倒序后只取最新 5 条为例子，数据库内部会根据查询计划选择不同策略：

1. **LIMIT 提前停止**：找到足够的结果后，不再向上层返回多余行。
2. **分页读取**：使用 `LIMIT` 和 `OFFSET`，或者使用上一页最后一条记录作为下一页起点。
3. **外部排序**：内存不够时，把临时排序数据写到硬盘，再分批合并。
4. **索引**：如果已经有适合的索引，可以直接按索引顺序找到最新记录，不必全表排序。

具体采用哪种方案由数据库的查询优化器决定。应用开发者需要做的是写清楚查询条件、限制返回数量，并为高频查询设计合适的索引。

## 索引：给常用查询建一张目录

没有索引时，数据库可能逐行检查 `films` 表，直到找到所有符合条件的记录。这叫全表扫描。

如果最常见的操作是按创建时间取最新记录，可以提前建立索引

索引有点像字典前面的检字表：用额外空间维护一个有序目录，查询时可以更快定位。它不是免费的：

- 索引会占用额外磁盘空间。
- 插入、更新和删除时也要维护索引，写入会变慢。
- 不要为每一列都建立索引，应根据真实查询和数据规模选择。

SQLite 等数据库底层通常使用 B 树一类的数据结构来实现索引。

## 用可视化工具查看 `.db` 文件

SQLite 数据库本质上是一个文件，但不建议用普通文本编辑器修改它。以下工具可以打开表、查看数据、运行 SQL：

- **DB Browser for SQLite**：轻量、免费，适合刚开始学习。
- **DBeaver Community**：支持多种数据库，适合以后切换到服务式数据库。